# VeronaCard Next‑POI Prediction – Metrics & Exploratory Analysis

*Generated automatically on 2025-06-25 13:01 UTC*

In [1]:
from pathlib import Path
import ast
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import re
import unicodedata
import io 

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True

In [2]:
# CELLA 2: DEFINIZIONE DI TUTTE LE FUNZIONI HELPER

print("Funzioni helper definite.")

def safe_parse_prediction(x):
    """
    Parsing robusto per la colonna 'prediction'.
    Gestisce diversi formati: stringhe, liste già parsate, JSON, etc.
    """
    if pd.isna(x) or x is None:
        return []
    
    # Se è già una lista
    if isinstance(x, list):
        return x
    
    # Se è una stringa
    if isinstance(x, str):
        # Rimuovi spazi extra
        x = x.strip()
        
        # Se sembra JSON
        if (x.startswith('[') and x.endswith(']')):
            try:
                parsed = ast.literal_eval(x)
                if isinstance(parsed, list):
                    return parsed
            except (ValueError, SyntaxError):
                pass
                
            # Prova con json.loads
            try:
                parsed = json.loads(x)
                if isinstance(parsed, list):
                    return parsed
            except json.JSONDecodeError:
                pass
        
        # Se contiene delle virgole, prova a fare split
        if ',' in x:
            # Rimuovi eventuali parentesi quadre
            clean_x = x.strip('[]')
            # Split e clean
            items = [item.strip().strip('"').strip("'") for item in clean_x.split(',')]
            # Filtra elementi vuoti
            items = [item for item in items if item]
            if items:
                return items
        
        # Se è una singola stringa, ritornala come lista
        if x:
            return [x]
    
    # Fallback: converti in stringa e prova di nuovo
    return safe_parse_prediction(str(x))

def clean_poi_name(s: str) -> str:
    """
    Normalizza una stringa in un ID pulito e canonico.
    Gestisce case, accenti, punteggiatura e spazi.
    """
    if not isinstance(s, str):
        s = str(s)
        
    s = s.strip().lower()
    
    # 1. Normalizza accenti e diacritici (es. "Erbè" -> "erbe")
    s = ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )
    
    # 2. Rimuovi tutta la punteggiatura (es. "d'erbe" -> "derbe", "arena." -> "arena")
    s = re.sub(r'[^\w\s]', '', s)
    
    # 3. Normalizza spazi interni (es. "piazza  erbe" -> "piazza erbe")
    s = re.sub(r'\s+', ' ', s).strip()
    
    return s

def build_alias_map(csv_path="../data/verona/vc_site.csv") -> dict:
    """
    Carica il CSV dei POI e crea una mappa da QUALSIASI
    variante di nome (name, name_short) al name_short canonico e pulito.
    """
    print(f"Costruzione Mappa Alias da '{csv_path}'...")
    try:
        df_poi = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"ERRORE: File '{csv_path}' non trovato.")
        print("Continuo senza Mappa Alias (precisione inferiore).")
        return {}
    
    alias_map = {}
    
    for _, row in df_poi.iterrows():
        # L'ID canonico è il name_short pulito
        canonical_id = clean_poi_name(row['name_short'])
        
        # Mappa il 'name' pulito all'ID canonico
        # Esempio: "basilica di san zeno" -> "san zeno"
        clean_name = clean_poi_name(row['name'])
        alias_map[clean_name] = canonical_id
        
        # Mappa il 'name_short' pulito a se stesso (per sicurezza)
        # Esempio: "san zeno" -> "san zeno"
        alias_map[canonical_id] = canonical_id

    print(f"Mappa Alias costruita. {len(alias_map)} varianti mappate a {len(df_poi)} POI unici.")
    return alias_map

def poi_id(x, poi_alias_map):
    """
    Converte 'x' in un identificatore hashable comparabile con ground-truth,
    utilizzando la Mappa di Alias.
    """
    # 1. Gestione Nodi
    if x is None or (np.isscalar(x) and pd.isna(x)):
        return str(x)

    # 2. Gestione Ricorsiva (per Liste)
    if isinstance(x, (list, tuple)):
        # --- CORREZIONE ---
        # Passa la mappa anche alla chiamata ricorsiva
        return tuple(poi_id(e, poi_alias_map) for e in x)
        # --- FINE CORREZIONE ---

    # 3. Gestione Dizionari
    if isinstance(x, dict):
        for key in ('poi', 'poi_id', 'name', 'id', 'title'):
            if key in x:
                # --- CORREZIONE ---
                # Passa la mappa anche alla chiamata ricorsiva
                return poi_id(x[key], poi_alias_map) 
                # --- FINE CORREZIONE ---
        try:
            return json.dumps(x, sort_keys=True)
        except (TypeError, ValueError):
            return str(x)
            
    # 4. Gestione Stringhe (IL CUORE DELLA LOGICA)
    #    Applica la pulizia e poi la mappa di alias
    s_clean = clean_poi_name(str(x))
    
    # Cerca nella mappa; se non trova, usa la stringa pulita
    return poi_alias_map.get(s_clean, s_clean)

def calculate_hit_at_k(predictions, ground_truth, k):
    """Calculate hit@k metric"""
    if not isinstance(predictions, list) or len(predictions) == 0:
        return False
    return ground_truth in predictions[:k]

def calculate_reciprocal_rank(predictions, ground_truth):
    """Calculate reciprocal rank"""
    if not isinstance(predictions, list) or len(predictions) == 0:
        return 0.0
    try:
        rank = predictions.index(ground_truth) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0

Funzioni helper definite.


In [3]:
# CELLA 3: LOOP DI ESECUZIONE AUTOMATICA

# --- 1. CONFIGURAZIONE ---
print("Inizio esecuzione automatica...")

# Definisci le tue variabili
ANCHOR_POINTS = ["middle", "penultimate"]
MODELS = [
    "deepseek-coder_33b",
    "llama3.1_8b",
    "qwen2.5_7b",
    "qwen2.5_14b",
    "mistral_7b",
    "mixtral_8x7b"
]
STRATEGIES = [
    "base_version",
    "with_geom",
    "with_geom_time",
    "with_geom_time_best_cluster",
    "with_geom_time_cluster"
]
EXCLUDE_FOLDERS = {"clustering_analysis", "DEV", "baseline_heuristics"}

# Definisci i path di base
BASE_RESULTS_DIR = Path("../results")
BASE_OUTPUT_DIR = Path("singole_metriche")

# Costruisci la mappa di alias UNA VOLTA SOLA
POI_ALIAS_MAP = build_alias_map("../data/verona/vc_site.csv")

# --- 2. LOOP PRINCIPALE ---

for anchor in ANCHOR_POINTS:
    for model in MODELS:
        for strategy in STRATEGIES:
            
            # Costruisci i path dinamici
            input_dir = BASE_RESULTS_DIR / anchor / model / strategy
            input_glob = str(input_dir / "*_pred_*.csv")
            metrics_output_path = BASE_OUTPUT_DIR / anchor / model / strategy
            
            print("\n" + "="*80)
            print(f"Processing: {anchor} / {model} / {strategy}")
            print(f"Input: {input_dir}")
            print(f"Output: {metrics_output_path}")
            print("="*80)

            # --- 3. ESECUZIONE DELLA PIPELINE ---
            # (Questo è il codice delle tue vecchie celle, ora dentro il loop)
            
            # --- Inizio Logica (da 7ac05b72) ---
            csv_files = sorted(glob.glob(input_glob))
            if not csv_files:
                print(f"⚠️ NESSUN FILE TROVATO. Salto.")
                continue
                
            metrics_output_path.mkdir(parents=True, exist_ok=True)
            
            dfs = []
            for fp_str in csv_files:
                fp = Path(fp_str)
                # print(f"  Processing {fp.name}...") # Rimosso per pulizia output
                try:
                    df = pd.read_csv(fp)
                except pd.errors.ParserError:
                    print(f"  Warning: Parser error in {fp}, skipping...")
                    continue
                
                year_match = re.search(r'dati_(\d{4})', fp.name) or re.search(r'veronacard_(\d{4})', fp.name)
                year = int(year_match.group(1)) if year_match else 2014
                df['year'] = year
                df['prediction_list'] = df['prediction'].apply(safe_parse_prediction)
                df = df[df['prediction_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)]
                dfs.append(df)

            if not dfs:
                print(f"⚠️ NESSUN DATO VALIDO TROVATO. Salto.")
                continue

            df_all = pd.concat(dfs, ignore_index=True)
            print(f"✅ Dati Caricati: {len(df_all):,} righe da {len(csv_files)} files")
            
            # --- Inizio Logica (da 0bc35347) ---
            print("Normalizing predictions and ground truth...")
            df_all['prediction_norm'] = df_all['prediction_list'].apply(
                lambda lst: [poi_id(e, POI_ALIAS_MAP) for e in lst] if isinstance(lst, list) else []
            )
            df_all['ground_truth_norm'] = df_all['ground_truth'].apply(lambda x: poi_id(x, POI_ALIAS_MAP))
            
            # --- Inizio Logica (da k1zvreptzs) ---
            print("Calculating evaluation metrics...")
            df_all['hit@1'] = df_all.apply(lambda row: calculate_hit_at_k(row['prediction_norm'], row['ground_truth_norm'], 1), axis=1)
            df_all['hit@5'] = df_all.apply(lambda row: calculate_hit_at_k(row['prediction_norm'], row['ground_truth_norm'], 5), axis=1)
            df_all['rr'] = df_all.apply(lambda row: calculate_reciprocal_rank(row['prediction_norm'], row['ground_truth_norm']), axis=1)

            # --- Inizio Logica (da 87936f47) ---
            # (Validazione metrica globale, utile per il log)
            metrics_global = {
                "Top-1 Accuracy": df_all["hit@1"].mean(),
                "Top-5 Hit Rate": df_all["hit@5"].mean(),
                "MRR": df_all["rr"].mean(),
            }
            print(f"  Hit@1: {metrics_global['Top-1 Accuracy']:.3f}, Hit@5: {metrics_global['Top-5 Hit Rate']:.3f}, MRR: {metrics_global['MRR']:.3f}")
            
            # --- Inizio Logica (da 3b12f0f6) ---
            by_year = (
                df_all
                .groupby('year')
                .agg(
                    top1=('hit@1', 'mean'),
                    hit5=('hit@5', 'mean'),
                    mrr=('rr', 'mean'),
                    n=('card_id', 'size')
                )
                .reset_index()
                .sort_values('year')
            )

            # --- Inizio Logica (da 66808ca3) ---
            print("Calculating per-year coverage...")
            coverage_by_year = []
            
            ground_truth_pois_set = set(df_all["ground_truth_norm"].unique())
            ground_truth_pois_clean = {poi for poi in ground_truth_pois_set if poi and str(poi) != 'nan' and str(poi) != 'None'}
            total_catalog_size = len(ground_truth_pois_clean)

            if total_catalog_size == 0:
                print("  ATTENZIONE: Catalogo totale vuoto.")
            
            for year in sorted(df_all['year'].unique()):
                df_year = df_all[df_all['year'] == year]
                predicted_pois_year = set()
                for preds in df_year["prediction_norm"]:
                    if isinstance(preds, list):
                        predicted_pois_year.update(preds)
                predicted_pois_year_clean = {poi for poi in predicted_pois_year if poi and str(poi) != 'nan' and str(poi) != 'None'}
                
                numerator_raw = len(predicted_pois_year_clean)
                intersection_pois_year = predicted_pois_year_clean.intersection(ground_truth_pois_clean)
                numerator_true_coverage = len(intersection_pois_year)

                true_coverage, diversity_ratio = 0.0, 0.0
                if total_catalog_size > 0:
                    true_coverage = numerator_true_coverage / total_catalog_size
                    diversity_ratio = numerator_raw / total_catalog_size
                    
                coverage_by_year.append({
                    'year': year,
                    'coverage': true_coverage,
                    'diversity_ratio': diversity_ratio,
                    'n_predicted_unique_year': numerator_raw
                })

            coverage_df = pd.DataFrame(coverage_by_year)
            by_year = by_year.merge(coverage_df, on='year', how='left')
            print(f"  Catalogo Globale: {total_catalog_size} POIs")

            # --- Inizio Logica (da lt5vo9wiuj) ---
            print("Exporting CSV files...")
            
            # Top 1
            top1_data = by_year[['year', 'top1']].copy()
            top1_data['top1_percent'] = (top1_data['top1'] * 100).round(2)
            top1_data = top1_data[['year', 'top1_percent']]
            top1_data.columns = ['Year', 'Top-1 Accuracy (%)']
            top1_data.to_csv(metrics_output_path / 'top1_metrics_canva.csv', index=False)

            # Hit 5
            hit5_data = by_year[['year', 'hit5']].copy()
            hit5_data['hit5_percent'] = (hit5_data['hit5'] * 100).round(2)
            hit5_data = hit5_data[['year', 'hit5_percent']]
            hit5_data.columns = ['Year', 'Top-5 Hit Rate (%)']
            hit5_data.to_csv(metrics_output_path / 'hit5_metrics_canva.csv', index=False)

            # MRR
            mrr_data = by_year[['year', 'mrr']].copy()
            mrr_data['mrr_percent'] = (mrr_data['mrr'] * 100).round(2)
            mrr_data = mrr_data[['year', 'mrr_percent']]
            mrr_data.columns = ['Year', 'mrr Hit Rate (%)']
            mrr_data.to_csv(metrics_output_path / 'mrr_metrics_canva.csv', index=False)

            # Coverage
            coverage_data = by_year[['year', 'coverage']].copy()
            coverage_data['coverage_percent'] = (coverage_data['coverage'] * 100).round(2)
            coverage_data = coverage_data[['year', 'coverage_percent']]
            coverage_data.columns = ['Year', 'Coverage (%)']
            coverage_data.to_csv(metrics_output_path / 'coverage_metrics_canva.csv', index=False)

            # Diversity
            diversity_data = by_year[['year', 'diversity_ratio']].copy()
            diversity_data['diversity_percent'] = (diversity_data['diversity_ratio'] * 100).round(2)
            diversity_data = diversity_data[['year', 'diversity_percent']]
            diversity_data.columns = ['Year', 'Diversity Ratio (%)']
            diversity_data.to_csv(metrics_output_path / 'diversity_metrics_canva.csv', index=False)

            # --- Inizio Logica (da 5ycc48tvxtw) ---
            # Combined
            combined_data = by_year[['year', 'top1', 'hit5', 'mrr', 'coverage', 'diversity_ratio']].copy()
            combined_data['top1_percent'] = (combined_data['top1'] * 100).round(2)
            combined_data['hit5_percent'] = (combined_data['hit5'] * 100).round(2)
            combined_data['mrr_percent'] = (combined_data['mrr'] * 100).round(2)
            combined_data['coverage_percent'] = (combined_data['coverage'] * 100).round(2)
            combined_data['diversity_percent'] = (combined_data['diversity_ratio'] * 100).round(2)

            combined_data = combined_data[['year', 'top1_percent', 'hit5_percent', 'mrr_percent', 'coverage_percent', 'diversity_percent']]
            combined_data.columns = ['Year', 'Top-1 Accuracy (%)', 'Top-5 Hit Rate (%)', 'MRR (%)', 'Coverage (%)', 'Diversity Ratio (%)']
            combined_data.to_csv(metrics_output_path / 'combined_metrics_canva.csv', index=False)

            # Long Format
            long_format = pd.concat([
                combined_data[['Year', 'Top-1 Accuracy (%)']].rename(columns={'Top-1 Accuracy (%)': 'Value'}).assign(Metric='Top-1 Accuracy'),
                combined_data[['Year', 'Top-5 Hit Rate (%)']].rename(columns={'Top-5 Hit Rate (%)': 'Value'}).assign(Metric='Top-5 Hit Rate'),
                combined_data[['Year', 'MRR (%)']].rename(columns={'MRR (%)': 'Value'}).assign(Metric='MRR'),
                combined_data[['Year', 'Coverage (%)']].rename(columns={'Coverage (%)': 'Value'}).assign(Metric='Coverage'),
                combined_data[['Year', 'Diversity Ratio (%)']].rename(columns={'Diversity Ratio (%)': 'Value'}).assign(Metric='Diversity Ratio')
            ])
            # (Mancava l'export di long_format nella tua cella, lo aggiungo)
            long_format.to_csv(metrics_output_path / 'metrics_long_format_canva.csv', index=False)

            print(f"✅ Esportazione completata per: {metrics_output_path}")

print("\n" + "🏁" * 20)
print("ESECUZIONE AUTOMATICA COMPLETATA!")
print(f"Tutti i risultati sono stati salvati in: {BASE_OUTPUT_DIR.resolve()}")

Inizio esecuzione automatica...
Costruzione Mappa Alias da '../data/verona/vc_site.csv'...
Mappa Alias costruita. 39 varianti mappate a 22 POI unici.

Processing: middle / deepseek-coder_33b / base_version
Input: ../results/middle/deepseek-coder_33b/base_version
Output: singole_metriche/middle/deepseek-coder_33b/base_version
✅ Dati Caricati: 611,728 righe da 12 files
Normalizing predictions and ground truth...
Calculating evaluation metrics...
  Hit@1: 0.000, Hit@5: 0.005, MRR: 0.002
Calculating per-year coverage...
  Catalogo Globale: 22 POIs
Exporting CSV files...
✅ Esportazione completata per: singole_metriche/middle/deepseek-coder_33b/base_version

Processing: middle / deepseek-coder_33b / with_geom
Input: ../results/middle/deepseek-coder_33b/with_geom
Output: singole_metriche/middle/deepseek-coder_33b/with_geom
✅ Dati Caricati: 551,611 righe da 12 files
Normalizing predictions and ground truth...
Calculating evaluation metrics...
  Hit@1: 0.050, Hit@5: 0.404, MRR: 0.157
Calculatin